# Minimum-motion equiluminance (Anstis & Cavanagh 1983): analysis

Reads one run folder (`*_trials.csv`) and computes the paper's measures.

- **min-motion-constant** (Experiment 1, Fig. 2): percentage of left / right / neither reports per test luminance, the point of subjective equality (PSE) and the width of the "neither" band.
- **min-motion-adjust / min-flicker-adjust** (Experiments 2-3, Fig. 3-4): mean and SE of the settings; set `ANALYSIS_FLICKER_RUN` to a uniform-field flicker run to express motion settings as luminous efficiency relative to the standard flicker match.

Set `ANALYSIS_RUN` to a run folder; by default the newest run under `data/` or `data-rehearsal/`.

**Units.** Luminances are the cd/m² the task computed from its photometry file. Check `luminance_basis`: with an unmeasured file they are nominal, device-specific values. Even when measured, a PSE is this observer's match for this stimulus on this display, not a general isoluminance. A simulated run is a software check, not data.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

from anstiscavanagh1983 import analysis as A

HERE = Path.cwd() if (Path.cwd() / 'run.py').exists() else Path.cwd().parent
RUN = os.environ.get('ANALYSIS_RUN')
if RUN is None:
    runs = sorted(HERE.glob('data*/**/*_trials.csv'), key=lambda p: p.stat().st_mtime)
    RUN = runs[-1].parent if runs else None
FLICKER_RUN = os.environ.get('ANALYSIS_FLICKER_RUN')  # optional: a min-flicker-adjust run, field uniform
print('run:', RUN)
trials = A.load_trials(RUN) if RUN else pd.DataFrame()
task = trials['task'].iloc[0] if len(trials) else None
print('task:', task, '| trials:', len(trials))
if len(trials):
    print('luminance basis:', trials['luminance_basis'].iloc[0])

## Experiment 1: constant stimuli

In [ ]:
if task == 'min-motion-constant':
    summ = A.constant_summary(trials)
    display(summ.round(3))
    pse = A.pse_crossing(summ)
    fit = A.fit_ordered_probit(trials)
    lo, hi, peak = A.neither_half_max(summ)
    ref = trials['reference_cd_m2'].iloc[0]
    print(f"reference ({trials['reference_colour'].iloc[0]}): {ref:g} cd/m2")
    print(f'PSE, crossing of test-lighter and reference-lighter reports: {pse:.3f} cd/m2')
    print(f"PSE, ordered-probit fit: {fit['pse_cd_m2']:.3f} cd/m2 (sigma {fit['sigma_ln']:.3f}, "
          f"delta {fit['delta_ln']:.3f} in ln units, n = {fit['n']})")
    print(f'neither band at half maximum: {lo:.3f} to {hi:.3f} cd/m2 (peak at {peak:.3f})')
    print(f'indifference half-width as colour-grating contrast: {A.indifference_contrast(summ, pse):.4f}'
          ' (paper: 0.016 for red/green, 0.028 for blue/yellow)')

In [ ]:
if task == 'min-motion-constant':
    coded = A.code_reports(trials)
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.4), sharey=True)
    right_steps = coded[coded['step'] == 1]
    if len(right_steps):
        s = A.constant_summary(right_steps)
        axes[0].plot(s['test_cd_m2'], 100 * s['p_left'], 'o-', mfc='white', color='k', label='left')
        axes[0].plot(s['test_cd_m2'], 100 * s['p_right'], 'o-', color='k', label='right')
        axes[0].plot(s['test_cd_m2'], 100 * s['p_neither'], '^--', mfc='white', color='0.4', label='neither')
    axes[0].set_title('Steps rightward (Fig. 2 layout)')
    axes[1].plot(summ['test_cd_m2'], 100 * summ['p_reference_lighter'], 'o-', mfc='white', color='k',
                 label='reference lighter')
    axes[1].plot(summ['test_cd_m2'], 100 * summ['p_test_lighter'], 'o-', color='k', label='test lighter')
    axes[1].plot(summ['test_cd_m2'], 100 * summ['p_neither'], '^--', mfc='white', color='0.4', label='neither')
    if pse == pse:
        axes[1].axvline(pse, color='#7f1d1d', lw=1)
    axes[1].set_title('Both step directions, by implied lighter colour')
    for ax in axes:
        ax.set_xlabel(f"{trials['test_colour'].iloc[0]} luminance (cd/m2, see basis)")
        ax.legend(frameon=False, fontsize=8)
    axes[0].set_ylabel('% of reports')
    fig.tight_layout()

## Experiments 2-3: adjustment

In [ ]:
if task in ('min-motion-adjust', 'min-flicker-adjust'):
    settings = A.adjust_summary(trials)
    display(settings.round(3))
    if FLICKER_RUN:
        flick = A.adjust_summary(A.load_trials(FLICKER_RUN))
        flick = flick[flick['layout'] == 'uniform']
        eff = A.relative_efficiency(settings, flick)
        display(eff.round(3))
        fig, ax = plt.subplots(figsize=(5, 3.4))
        xcol = 'spatial_freq_cpd' if eff['spatial_freq_cpd'].nunique() > 1 else 'rate_hz'
        for colour, g in eff.groupby('test_colour'):
            ax.plot(g[xcol], g['relative_efficiency'], 'o-', label=colour)
        ax.set_xscale('log', base=2)
        ax.set_xlabel(xcol)
        ax.set_ylabel('luminous efficiency / flicker match')
        ax.legend(frameon=False)